# 7.7 Regularization and Generalization

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook focuses on the gap between fitting the training data and performing well on new data. The examples emphasize overfitting controls rather than larger architectures.


## 7.7.1 Recognizing overfitting

### Why It Matters
Overfitting appears when training performance keeps improving while validation performance stops improving or worsens. A deliberately oversized model on a small dataset makes the pattern visible.


In [ ]:
import torch
from torch import nn
from sklearn.model_selection import train_test_split

torch.manual_seed(24)
X = torch.randn(70, 12)
y = ((X[:, 0] - X[:, 1] + 0.3 * X[:, 2]) > 0).float().unsqueeze(1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=24)
model = nn.Sequential(nn.Linear(12, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
history = []
for _ in range(25):
    opt.zero_grad()
    train_loss = loss_fn(model(X_train), y_train)
    train_loss.backward()
    opt.step()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val)
    history.append((round(float(train_loss.item()), 4), round(float(val_loss.item()), 4)))
history[-5:]


### Interpretation
When validation loss decouples from training loss, you are seeing the practical meaning of poor generalization.


## 7.7.2 Dropout

### Why It Matters
Dropout randomly removes activations during training, forcing the network to rely less on any one hidden pathway.


In [ ]:
import torch
from torch import nn

torch.manual_seed(25)
model = nn.Sequential(nn.Linear(6, 6), nn.ReLU(), nn.Dropout(0.5), nn.Linear(6, 1))
x = torch.ones(4, 6)
model.train()
train_out = model(x)
model.eval()
eval_out = model(x)
{
    "train_mode_variation": train_out.detach().flatten().round(decimals=3).tolist(),
    "eval_mode_outputs": eval_out.detach().flatten().round(decimals=3).tolist(),
}


### Interpretation
Dropout acts as stochastic regularization during training but becomes deterministic at evaluation time.


## 7.7.3 Weight decay

### Why It Matters
Weight decay discourages large parameter values by adding a penalty through the optimizer. The effect is easiest to inspect through the weight norm.


In [ ]:
import copy
import torch
from torch import nn

torch.manual_seed(26)
X = torch.randn(100, 5)
y = ((X[:, 0] + X[:, 1]) > 0).float().unsqueeze(1)
base = nn.Sequential(nn.Linear(5, 12), nn.ReLU(), nn.Linear(12, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
results = {}
for label, wd in [("no_decay", 0.0), ("weight_decay", 0.1)]:
    model = copy.deepcopy(base)
    opt = torch.optim.Adam(model.parameters(), lr=0.03, weight_decay=wd)
    for _ in range(150):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    total_norm = sum(p.norm().item() for p in model.parameters())
    results[label] = {"loss": round(float(loss.item()), 4), "parameter_norm_sum": round(float(total_norm), 3)}
results


### Interpretation
Regularization is not free: it can improve generalization while slightly constraining how aggressively the model fits the training data.


## 7.7.4 Early stopping

### Why It Matters
Instead of training forever, early stopping keeps the model state from the point where validation performance was best.


In [ ]:
import copy
import torch
from torch import nn
from sklearn.model_selection import train_test_split

torch.manual_seed(27)
X = torch.randn(140, 4)
y = ((X[:, 0] - 0.7 * X[:, 2]) > 0).float().unsqueeze(1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=27)
model = nn.Sequential(nn.Linear(4, 20), nn.ReLU(), nn.Linear(20, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.04)
loss_fn = nn.BCELoss()
best_epoch, best_val, best_state = None, float('inf'), None
for epoch in range(20):
    opt.zero_grad()
    train_loss = loss_fn(model(X_train), y_train)
    train_loss.backward()
    opt.step()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val).item()
    if val_loss < best_val:
        best_epoch, best_val, best_state = epoch, val_loss, copy.deepcopy(model.state_dict())
{"best_epoch": best_epoch, "best_val_loss": round(float(best_val), 4), "saved_tensors": len(best_state)}


### Interpretation
Early stopping is a model-selection rule: it chooses the epoch that generalized best rather than the final epoch by default.


## 7.7.5 Data augmentation intuition

### Why It Matters
Augmentation creates additional plausible training examples by transforming existing ones. On image-like tensors, even a simple flip changes the sample while preserving the label idea.


In [ ]:
import torch
from torchvision import transforms

image = torch.arange(16, dtype=torch.float32).reshape(1, 4, 4)
augmenter = transforms.RandomHorizontalFlip(p=1.0)
flipped = augmenter(image)
{
    "original_first_row": image[0, 0].tolist(),
    "flipped_first_row": flipped[0, 0].tolist(),
}


### Interpretation
The model sees a legitimate variation of the same underlying pattern, which can improve robustness when real data vary similarly.


## 7.7.6 Compare overfit versus regularized models

### Why It Matters
This final subsection compares a plain network and a dropout-regularized network on the same small training split.


In [ ]:
import copy
import torch
from torch import nn
from sklearn.model_selection import train_test_split

torch.manual_seed(28)
X = torch.randn(120, 10)
y = ((X[:, 0] + X[:, 1] - X[:, 3]) > 0).float().unsqueeze(1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.35, random_state=28)
plain = nn.Sequential(nn.Linear(10, 40), nn.ReLU(), nn.Linear(40, 1), nn.Sigmoid())
regularized = nn.Sequential(nn.Linear(10, 40), nn.ReLU(), nn.Dropout(0.4), nn.Linear(40, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
results = {}
for name, model in [("plain", plain), ("dropout", regularized)]:
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    for _ in range(30):
        model.train()
        opt.zero_grad()
        train_loss = loss_fn(model(X_train), y_train)
        train_loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val).item()
    results[name] = {"train_loss": round(float(train_loss.item()), 4), "val_loss": round(float(val_loss), 4)}
results


### Interpretation
Regularization should be judged by validation behavior, not by whether it drives the training loss down faster.
